<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, on one report_date, for one client
(grain: report_date x client_hash_id x content_hash_id) in
fact_content_daily_performance, for month=2026-03 — a mid-panel month,
deliberately NOT the _sample table (which is June 2026, the sealed final
month I'm not allowed to develop label logic on).

In [ ]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {TABLES['fact_daily']}
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context (join/group only, never features): report_date, client_hash_id,
content_hash_id.

Features (observed, knowable before the review moment): gsc_impressions,
gsc_clicks, gsc_avg_position, sessions_ai, content_type (from dim_content).

Label/proxy: CTR gap = actual monthly CTR (gsc_clicks/gsc_impressions) minus
the expected CTR for that page's position tier, both computed from the SAME
month — this is Lane 4's target, a proxy for "under-capturing clicks for how
well this page ranks."

Excluded: any product-decision flags (health_score, priority_score,
action_type) — not shipped in this data anyway, so nothing to strip, but
naming it here as a discipline; also any report_date outside month=2026-03
for this notebook, to keep the feature window clean.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLES['fact_daily']}
""").df()

,n_rows,min_d,max_d
0,9841378,2026-03-01,2026-03-31


In [ ]:
before = con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily']}").fetchone()[0]
after = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['fact_daily']}
    WHERE ga4_data_available IS TRUE
""").fetchone()[0]
print(f"{before:,} rows total -> {after:,} rows with real GA4 data ({after/before:.1%})")

9,841,378 rows total -> 413,966 rows with real GA4 data (4.2%)


In [ ]:
feats = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions)                              AS impressions_month,
           SUM(gsc_clicks)                                   AS clicks_month,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_month,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_month,
           SUM(sessions_ai)                                  AS ai_sessions_month
    FROM {TABLES['fact_daily']}
    GROUP BY content_hash_id
    HAVING impressions_month >= 100
""").df()
feats.head()

,content_hash_id,impressions_month,clicks_month,ctr_month,avg_position_month,ai_sessions_month
0,content_7a105f548d9c6916,6523.0,7.0,0.001073,7.209549,0.0
1,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.307255,0.0
2,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.724039,0.0
3,content_a7da352b73b02668,4944.0,13.0,0.002629,7.244844,0.0
4,content_1855a661b4d36130,429.0,1.0,0.002331,4.499519,0.0


1. impressions_month — knowable at the decision moment because it's summed
   only from report_dates inside the month I'm reviewing, nothing future.
2. clicks_month — same: purely trailing observed counts.
3. ctr_month — a ratio of the two above, so equally "past-only."
4. avg_position_month — an average of daily positions already recorded.
5. ai_sessions_month — observed AI-referral sessions already logged for the month.


In [ ]:
import pandas as pd

feats["position_tier"] = pd.cut(feats["avg_position_month"],
                                 bins=[0,3,10,20,1000], labels=["1-3","4-10","11-20","21+"])
expected = feats.groupby("position_tier")["ctr_month"].transform("mean")
feats["ctr_gap"] = feats["ctr_month"] - expected
feats["is_underperforming"] = (feats["ctr_gap"] < -0.005).astype(int)


honest_cols = ["impressions_month", "avg_position_month", "ai_sessions_month"]
print(feats[honest_cols + ["is_underperforming"]].corr()["is_underperforming"])

impressions_month    NaN
avg_position_month   NaN
ai_sessions_month    NaN
is_underperforming   NaN
Name: is_underperforming, dtype: float64


/tmp/ipykernel_3792/65429766.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  expected = feats.groupby("position_tier")["ctr_month"].transform("mean")


In [ ]:
leaky_cols = honest_cols + ["ctr_month"]
print(feats[leaky_cols + ["is_underperforming"]].corr()["is_underperforming"])

impressions_month    NaN
avg_position_month   NaN
ai_sessions_month    NaN
ctr_month            NaN
is_underperforming   NaN
Name: is_underperforming, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice under-represents AI-referral behavior badly: warehouse-wide, only
~30k of 78.8M daily rows have any AI session data at all, so ai_sessions_month
is mostly zero here and shouldn't be trusted as a standalone signal for a
single mid-panel month. Also, march-2026 history depth differs per client
(unbalanced panel) — some clients may have little or no data in this exact
month, so this slice isn't evenly representative across all 70 clients.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.